# Basic Ticketing System Agent Demonstration

This notebook serves as a basic example of how to create a LangChain agent that interacts with the Bean Machine Ticketing System's OpenAPI specification. It demonstrates how to use LangChain's OpenAPI integration to build an agent that can reason over the API and make calls to it.

It also includes examples of how to connect to a websocket endpoint and how to import the support information into a retrieval system. The notebook is designed to be a *_starting point_* for building a more advanced agent that can handle the requirements of the capstone project.

You can use this notebook as a starting point for your own agent development, adapting it to your specific needs and extending its functionality as required. Or you can take inspiration from it and build your own agent from scratch.

## Key Features
- Using websockets to connect to a ticketing system.
- Retrieving and processing the OpenAPI specification.
- Creating a simple agent that can interact with the ticketing system.
- Importing the support information into a chroma vector store.


## Getting Started

### Start the Ticketing System
Before running this notebook, ensure that the ticketing system is running. You can do this by following the instructions in the capstone project.

### Install Dependencies
Make sure you have the required dependencies installed. You can do this by running the following command in your terminal:

In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install tiktoken --only-binary=:all:

In [ ]:
# Install required libraries
%pip install -qU \
    langchain==0.3.* \
    langchain_openai==0.3.* \
    langchain_community \
    unstructured[md]==0.17.* \
    langgraph==0.4.* \
    websockets==15.0.*

In [ ]:
import os
import getpass

os.environ['OPENAI_API_KEY'] = getpass.getpass("Enter your OpenAI API key: ")

## Subscribe to Updates using Websockets

Websockets are a two-way communication protocol that allows for real-time data updates between a client (like your Agent) and a server (like the ticketing system). They provide a way for your agent to receive real-time updates from the ticketing system, which is particularly useful for monitoring ticket status changes, such as when tickets are created, updated, or closed.

**Why use Websockets here?**
- They allow you to see ticket updates in real time, which is useful for responding to changes as they happen.
- They are more efficient than traditional HTTP requests, as they maintain a persistent connection and only require data to be sent when there is an update.
- This is more efficient than polling, which would require sending repeated requests to the server.

**How does this work in the notebook?**
- The code below sets up a WebSocket connection to the ticketing system.
- While the connection is open, any ticket updates from the server will be printed in the notebook output.
- You can use this to watch how the agent (or other users) interact with tickets live.

**Note:**
- The WebSocket connection runs until you manually interrupt the kernel (stop the cell).
- If you have issues with Websockets, you can use HTTP Polling (periodically checking for updates) or trigger the agent manually (see next section for details).


The following cell contains code to set up the WebSocket connection. Run it and try creating or updating a ticket in the ticketing system. You should see the updates appear in real time in the notebook output. Once you are done, you can stop the WebSocket connection by interrupting the kernel (e.g., using the stop button in Jupyter).

In [ ]:
import websockets

# URL for the WebSocket server (make sure the ticketing system is running)
WS_URL = "ws://localhost:3000/ws"

# This async function connects to the WebSocket and listens for ticket updates
async def listen_for_ticket_updates():
    print("Starting connection")
    # Establish a connection to the WebSocket server
    async with websockets.connect(WS_URL) as websocket:
        print("WebSocket connection established.")
        try:
            # Keep listening for messages from the server
            while True:
                message = await websocket.recv()  # Wait for a new message
                print(f"Ticket update received! Ticket ID: {message}")  # Print the update
        except websockets.ConnectionClosed:
            print("WebSocket connection closed.")
        except Exception as e:
            print(f"WebSocket error: {e}")

# To run the async function in a notebook cell, use 'await' (Jupyter supports this)
await listen_for_ticket_updates()

### Alternative: HTTP Polling for Ticket Updates

If you cannot use Websockets, you can still check for ticket updates by periodically sending HTTP requests to the server. This is called "polling." While not real-time, it is simple and reliable.

**How HTTP Polling Works:**
- The notebook sends a request to the server at regular intervals (e.g., every 10 seconds) to check for new or updated tickets.
- If there are updates, the server responds with the latest ticket data.

**Sample Steps:**
1. Set up a loop in your notebook that sends a GET request to the ticketing system's endpoint (e.g., `/api/tickets`).
2. Wait a few seconds between each request (using `time.sleep`).
3. Print or process any new updates.

In [ ]:
import time
import requests

while True:
    response = requests.get("http://localhost:3000/api/tickets")
    tickets = response.json()
    print(tickets)  # Or process as needed
    time.sleep(10)  # Wait 10 seconds before polling again

### Alternative: Manually Triggering the Agent

You can also interact with the ticketing system by running specific notebook cells to trigger the agent's actions, without waiting for updates.

**How to Manually Trigger the Agent:**
1. Run the code cell that creates or updates a ticket (or any agent action you want to test).
2. Observe the output in the notebook to see the result.
3. Repeat as needed for different actions or tickets.

**When to Use Manual Triggering:**
- When you want to test a specific scenario or agent behavior.
- When you do not need real-time updates and just want to see the result of a single action.


## Load and Inspect the Ticketing System OpenAPI Schema

Before interacting with the ticketing system programmatically, it's important to understand what actions (endpoints) are available and how to use them. The OpenAPI schema is a standardized description of the API, listing all endpoints, their parameters, and expected responses.

**Why are we doing this?**
- To discover what operations (like creating, updating, or listing tickets) the API supports.
- To see what data you need to send and what you can expect in return.
- To help the agent (and you) interact with the API correctly and efficiently.

**What to look for in the code results:**
- The list of available servers (where the API is hosted).
- The API's general description.
- The endpoints (paths) you can call, such as `/api/tickets`, `/api/tickets/{id}`, etc.

**How to use this information:**
- Use the endpoint list to decide which actions your agent or code should perform.
- Refer to the parameters and descriptions to format your requests correctly.
- This schema will be used by the LangChain agent to automatically understand and interact with the ticketing system.

Run the next cell to load and print the OpenAPI schema details.

In [ ]:
from langchain_community.agent_toolkits.openapi.spec import reduce_openapi_spec
import requests

# Load the OpenAPI specification from the running ticketing system
root = "http://localhost:3000"
api_spec_url = f"{root}/api/docs/openapi.json"

# Download and parse the OpenAPI spec
response = requests.get(api_spec_url)
data = response.json()
data['servers'] = [{'url': root}]
openapi_spec = reduce_openapi_spec(data, dereference=False)

# Show the OpenAPI spec details
print('Servers:', openapi_spec.servers)
print('Descriptions:', openapi_spec.description)
print('Endpoints:')
for endpoint in openapi_spec.endpoints:
    print(endpoint)

## Create and Test the LangChain OpenAPI Agent

This cell creates a LangChain agent that can interact with the ticketing system using the OpenAPI schema we just loaded. The Agent uses a built-in OpenAPI agent from LangChain, which simplifies the creation process. It also includes an example query to demonstrate how the agent can interact with the ticketing system.

Run the next cell to create the agent and test it with a sample query. The agent will use the OpenAPI schema to understand how to interact with the ticketing system.

In [ ]:
from langchain_community.utilities.requests import RequestsWrapper
from langchain_community.agent_toolkits.openapi import planner
from langchain_openai import ChatOpenAI

requests_wrapper = RequestsWrapper()
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.0)

agent = planner.create_openapi_agent(
    api_spec=openapi_spec,
    requests_wrapper=requests_wrapper,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    handle_parsing_errors=True,
    allow_operations=['GET', 'POST', 'PUT', 'PATCH', 'DELETE']
)

# Example: Close a ticket by ID 
result = agent.invoke('Close ticket 1a2b3c4d-0001-0000-0000-000000000001 category to "Maintenance"')
print(result)

### Basic Categorization Agent

The following cell is a basic example of how to create an agent that can categorize tickets based on their content. This agent will listen for ticket updates via WebSocket and categorize each ticket into one of the predefined categories: "Mechanical", "Quality", "Maintenance", or "Technical".

Run the cell below to create the agent and start listening for ticket updates. Once the cell is running, create a new ticket in the ticketing system, and the agent will categorize it based on its content.

In [ ]:
import json

# This async function connects to the WebSocket and listens for ticket updates
# Once a ticket update is received, it yields it for processing.
async def listen_for_ticket_updates():
    print("Starting connection")
    # Establish a connection to the WebSocket server
    async with websockets.connect(WS_URL) as websocket:
        print("WebSocket connection established.")
        try:
            # Keep listening for messages from the server
            while True:
                message = await websocket.recv()  # Wait for a new message
                yield json.loads(message)
        except websockets.ConnectionClosed:
            print("WebSocket connection closed.")
        except Exception as e:
            print(f"WebSocket error: {e}")

# To run the async function in a notebook cell, use 'await' (Jupyter supports this)
async for message in listen_for_ticket_updates():
    ticket_id = message.get('ticketId')
    update_type = message.get('updateType')
    
    if update_type == 'created':
        print(f'Categorizing ticket: {ticket_id}')
        agent.invoke(f"""
Based only on the ticket information, categorize the ticket into one of the following categories:
                
- Mechanical
- Quality
- Maintenance
- Technical
                
Ticket ID: {ticket_id}
""".strip())

## Basic RAG System
The follow cell sets up a basic Retrieval-Augmented Generation (RAG) retriever for the support information. This allows the agent to access relevant support documents when answering user queries, enhancing its ability to provide accurate and helpful responses. It does this by:

1. Loading the support documents from a specified directory.
2. Creating a vector store to index the documents.
3. Demonstrating how to use the retriever to get relevant information based on a user query.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

loader = DirectoryLoader("./support-info")
docs = loader.load()

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(docs)

vector_store.as_retriever().invoke("Machine won't start.")